### Step 0: Import + Path

In [4]:
import os, glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import pyarrow.parquet as pq

TEST_CSV = "C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test.csv"
SPEC_DIR = "C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms"  # in your case you uploaded parquet files directly here
WINDOW_SECONDS = 10.0


### Step1: Load test.csv & Inspect

In [5]:
df = pd.read_csv(TEST_CSV)
print("shape:", df.shape)
print("columns:", df.columns.tolist())
display(df.head(10))

shape: (10680, 17)
columns: ['eeg_id', 'eeg_sub_id', 'eeg_label_offset_seconds', 'spectrogram_id', 'spectrogram_sub_id', 'spectrogram_label_offset_seconds', 'label_id', 'patient_id', 'expert_consensus', 'seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote', 'fold', 'split']


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,fold,split
0,1202099836,0,0.0,1353070,0,0.0,3051612434,34554,Other,0,0,5,0,0,9,0,test
1,8071080,0,0.0,2593634,0,0.0,92643023,2944,GRDA,0,0,0,0,3,0,0,test
2,8071080,1,10.0,2593634,1,10.0,2540496740,2944,GRDA,0,0,0,0,3,0,0,test
3,8071080,2,14.0,2593634,2,14.0,1091227536,2944,GRDA,0,0,0,0,3,0,0,test
4,8071080,3,18.0,2593634,3,18.0,1273898793,2944,GRDA,0,0,0,0,3,0,0,test
5,989810287,0,0.0,2843061,0,0.0,116172961,13521,Other,0,1,0,0,0,4,0,test
6,989810287,1,6.0,2843061,1,6.0,1705072321,13521,Other,0,1,0,0,0,4,0,test
7,989810287,2,8.0,2843061,2,8.0,3784110610,13521,Other,0,1,0,0,0,4,0,test
8,2322119862,0,0.0,6249322,0,0.0,2214815956,13521,Other,0,0,0,0,0,1,0,test
9,443031384,0,0.0,9167937,0,0.0,3074802187,8674,Other,0,0,0,1,7,8,0,test


### Step2: See which .parquet files are available

In [6]:
parquets = sorted(glob.glob(os.path.join(SPEC_DIR, "*.parquet")))
print("parquet count:", len(parquets))
print("first few:", parquets[:5])



parquet count: 1154
first few: ['C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms\\1003145721.parquet', 'C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms\\100332931.parquet', 'C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms\\1004024952.parquet', 'C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms\\1006186695.parquet', 'C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms\\1008148805.parquet']


**Step3: Read 1 parquet file inspect**

In [7]:
sample_path = parquets[30]
table = pq.read_table(sample_path)
print("rows, cols:", table.num_rows, table.num_columns)
print("first 5 columns:", table.schema.names[:5])
print("has 'time'?:", "time" in table.schema.names)

spec_df = table.to_pandas()
print("pandas shape:", spec_df.shape)
display(spec_df.head(20)) 

rows, cols: 318 401
first 5 columns: ['time', 'LL_0.59', 'LL_0.78', 'LL_0.98', 'LL_1.17']
has 'time'?: True
pandas shape: (318, 401)


,time,LL_0.59,LL_0.78,LL_0.98,LL_1.17,LL_1.37,LL_1.56,LL_1.76,LL_1.95,LL_2.15,...,RP_18.16,RP_18.36,RP_18.55,RP_18.75,RP_18.95,RP_19.14,RP_19.34,RP_19.53,RP_19.73,RP_19.92
0,1,25.760000,28.660000,31.809999,18.940001,17.219999,14.460000,11.00,10.93,6.31,...,0.01,0.01,0.00,0.00,0.00,0.01,0.01,0.01,0.01,0.01
1,3,26.219999,24.530001,24.309999,23.110001,10.430000,10.760000,8.88,5.98,4.07,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
2,5,22.580000,20.040001,25.790001,21.719999,13.190000,8.210000,6.03,2.94,2.65,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
3,7,26.730000,32.740002,35.299999,29.860001,21.180000,13.300000,6.47,4.69,3.92,...,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.00
4,9,20.129999,23.250000,24.570000,20.330000,14.920000,10.560000,7.00,7.02,4.57,...,0.02,0.02,0.01,0.00,0.01,0.01,0.00,0.00,0.01,0.01
5,11,15.690000,17.809999,20.799999,12.270000,15.660000,12.220000,8.49,9.99,4.29,...,0.01,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.01,0.01
6,13,22.600000,35.590000,35.919998,27.940001,28.280001,15.760000,5.99,5.03,3.44,...,0.01,0.01,0.01,0.01,0.00,0.00,0.00,0.01,0.00,0.00
7,15,29.510000,34.360001,42.349998,33.660000,15.590000,13.470000,10.03,4.31,2.52,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.00
8,17,17.090000,27.500000,24.650000,27.180000,20.410000,10.180000,7.50,6.07,4.06,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
9,19,21.280001,32.950001,37.689999,31.600000,28.549999,17.080000,10.53,6.54,5.27,...,0.01,0.01,0.00,0.00,0.01,0.01,0.01,0.01,0.01,0.00


**Step0: Loader function**

Loads a spectrogram → extracts a small time window around a given offset → cleans it → converts it into a format usable by a CNN.

In [8]:
import os
from time import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import IterableDataset, DataLoader

SPEC_DIR = "C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test_spectrograms" 
CLASSES = ["seizure", "lpd", "gpd", "lrda", "grda", "other"]

def load_spec_window_parquet(spec_id: int, offset_s: float, window_seconds: float = 10.0):

# 1. load parquet
    path = os.path.join(SPEC_DIR, f"{spec_id}.parquet")
    spec_df = pd.read_parquet(path)  

# 2. Separate time and frequency data
    time = spec_df["time"].to_numpy()
    freq_mat = spec_df.drop(columns=["time"]).to_numpy(dtype=np.float32)  # (T, F)

# 3. Finding Event Center of the 10 min window
#  offset_s is where the 10 min starts
    center_time = offset_s + 300.0  # 600 sec (10 min) window → center at +300 sec
    center_idx = int(np.argmin(np.abs(time - center_time)))

    dt = float(np.median(np.diff(time))) if len(time) > 1 else 1.0     # how many seconds per spectrogram row
    half_window = int(round((window_seconds / 2) / dt))                # Decide window size. 5s before, 5s after = 10 seconds total Window
    target_len = 2 * half_window                             # (10/2)/2 = 2.5 = 2*2 = 4rows. This is the fixed number of time steps your window should have, 4 rows  

# 4. Crop/Extract Window Around Event - Temporal windowing
    start = max(0, center_idx - half_window)
    end   = min(len(time), center_idx + half_window)
    window = freq_mat[start:end, :]  # (t, f)

# 5. Pad if near egdes add zeros to keep fixed size inputs
    if window.shape[0] < target_len:
        pad = target_len - window.shape[0]
        window = np.pad(window, ((0, pad), (0, 0)), mode="constant")

# 6. log(stability) + normalize(mean=0, std=1, helps training converge fatser)
    window = np.log1p(np.maximum(window, 0))
    window = (window - window.mean()) / (window.std() + 1e-6)     # μ = mean of this window, σ = standard deviation, 1e-6 avoids division by zero

    # CNN format: (C,H,W)
    # If you trained with (1, F, T), keep this:
    window = window.T  # originally (T, F) after transpose (F, T)
    return window[None, :, :]  # (1, F, T)


**Step1 — IterableDataset for test (streams rows)**  
IterableDataset streams data sequentially. It doesn’t require indexing. Good for large datasets that don’t fit in RAM  

In [ ]:
#custom PyTorch streaming dataset.
class HMSTestParquetStream(IterableDataset):
    def __init__(self, test_df: pd.DataFrame, window_seconds: float = 10.0): #saves test.csv
        self.df = test_df.reset_index(drop=True)                             #reset index to ensure it's 0,1,2,... for iteration
        self.window_seconds = window_seconds                                 #saves the time of each cropped window. 10s here

    # When PyTorch's DataLoader runs, it calls this function.
    def __iter__(self):
        for i in range(len(self.df)): #Go through test.csv row by row
            row = self.df.iloc[i]
            spec_id = int(row["spectrogram_id"])                       # which parquet file to open
            offset_s = float(row["spectrogram_label_offset_seconds"])  # where to crop
            label_id = int(row["label_id"])                            # needed to build submission later

            x = load_spec_window_parquet(spec_id, offset_s, window_seconds=self.window_seconds) #load 1 parquet window. x is (C,H,W)(1, freq_bins, time_steps)
            yield torch.tensor(x, dtype=torch.float32), label_id # yields 1 sample at a time to pytorch
            



            


**Step2— DataLoader + inference loop (your trained CNN)**

In [13]:
test = pd.read_csv("C:\\AIG_Sem2\\CapstoneProject\\Capstone_EEG_Harmful-Brain-Activity-classification\\test.csv")
test = test[test["split"] == "test"].reset_index(drop=True)  # if split exists

stream = HMSTestParquetStream(test, window_seconds=10.0)
loader = DataLoader(stream, batch_size=32, num_workers=2, pin_memory=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = ...  # load your trained CNN here
model.to(device).eval()

all_label_ids = []
all_probs = []

with torch.no_grad():
    for xb, label_ids in loader:
        xb = xb.to(device)
        logits = model(xb)                  # (B,6)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_label_ids.extend(label_ids)
        all_probs.append(probs)

all_probs = np.vstack(all_probs)

sub = pd.DataFrame(all_probs, columns=CLASSES)
sub.insert(0, "label_id", all_label_ids)
sub.to_csv("submission.csv", index=False)
print("saved submission.csv", sub.shape)

Functional Testing

In [13]:
spec_id = 1353070  # <-- change to a real one you have

x = load_spec_window_parquet(spec_id, offset_s=0.0, window_seconds=10.0)

print("2D spectrogram window extracted around a specific offset_s:", x)
print("Output type:", type(x))
print("Output dtype:", x.dtype)
print("Output shape:", x.shape)          # expected: (1, F, T)
print("min/max:", np.min(x), np.max(x))
print("mean/std:", np.mean(x), np.std(x))
print("any NaN?", np.isnan(x).any())     # check for NaN values
print("any Inf?", np.isinf(x).any())     # check for Infinite values

2D spectrogram window extracted around a specific offset_s: [[[ 1.8566979   2.0027997   1.7728685   1.8688245 ]
  [ 2.0375981   2.1834774   1.9746749   2.1034656 ]
  [ 1.950519    2.2116897   2.0475578   2.229855  ]
  ...
  [-0.82610816 -0.7902986  -0.81882    -0.74265563]
  [-0.84088165 -0.79733646 -0.82610816 -0.7764015 ]
  [-0.83346164 -0.81882    -0.83346164 -0.8044353 ]]]
Output type: <class 'numpy.ndarray'>
Output dtype: float32
Output shape: (1, 400, 4)
min/max: -0.910869 3.833187
mean/std: 1.4305114e-08 0.9999992
any NaN? False
any Inf? False


In [18]:
x = load_spec_window_parquet(spec_id, 0.0)
x_t = torch.from_numpy(x).float()
print(x_t.shape, x_t.dtype)

torch.Size([1, 400, 4]) torch.float32


In [20]:
spec_id = 1353070  # same one you tested
df = pd.read_parquet(os.path.join(SPEC_DIR, f"{spec_id}.parquet"))
time = df["time"].to_numpy()

print("len(time):", len(time))
print("time[0], time[-1]:", time[0], time[-1])
print("dt median:", np.median(np.diff(time)))
print("dt min/max:", np.min(np.diff(time)), np.max(np.diff(time)))

len(time): 300
time[0], time[-1]: 1 599
dt median: 2.0
dt min/max: 2 2
